In [1]:
import pandas as pd

import matplotlib.pyplot as plot
import matplotlib.dates as mdates
import seaborn as sns
import time
from datetime import datetime, timedelta, date
import teradata
import businessdays as bzd
from pandasql import sqldf
import pygwalker as pyg
from pivottablejs import pivot_ui

import warnings
warnings.simplefilter('ignore')


start = time.time() 
system = "TERADBAWS.sicredi.net"
user = "P_USR_DL_GESTAO_RECURSOS"
passw = "JAW0b71329aadade35365afc6X"
udaExec = teradata.UdaExec (appName="TESTE", version="1.0", logConsole=False)
session = udaExec.connect(method="odbc", system=system,
        username=user, password=passw, Authentication="TDNEGO");



select_feriados = "Select DT from dl_Gestao_recursos.bas_cad_fer where tipo in ('3','2')"
feriados = pd.read_sql(select_feriados, session)
cal = bzd.Calendar(holidays=feriados.DT,weekdays=['Sunday','Saturday'])

data_inicial = cal.offset(datetime.now(),-60 )###DATA PADRÃO D-60
data_inicial = data_inicial.strftime('%Y-%m-%d') ###DATA PADRÃO D-90

data_final = cal.offset(datetime.now(),-1  )###DATA PADRÃO D-1
data_final = data_final.strftime('%Y-%m-%d') ###DATA PADRÃO D-1


In [2]:

Select = f"""select pco.dt
, pco.contrato
, pco.fuvct_cd
, pco.vl_preco
, pco.vl_ajuste_dia_anterior
, vcto.dt_limite_negociacao
, '' || extract(month from vcto.dt_limite_negociacao) as Mes_reuniao
, (vcto.vl_preco_exerc - 100)*100 as CUT_HIKE
, case when vcto.vl_preco_exerc = 100 then '' else case when vcto.vl_preco_exerc < 100 then '-' else '+' end end || to_char(abs((vcto.vl_preco_exerc - 100)*100), '999') as CUT_HIKE_2
, cast(vcto.dt_limite_negociacao as date format 'MMM-YY') || case when vcto.vl_preco_exerc = 100 then '' else case when vcto.vl_preco_exerc < 100 then '  -' else '  +' end end || to_char(abs((vcto.vl_preco_exerc - 100)*100), '999') as ROTULO
, case when vcto.vl_preco_exerc = 100 then '' else case when vcto.vl_preco_exerc < 100 then '-' else '+' end end as teste

, DENSE_RANK() OVER (PARTITION BY pco.dt ORDER BY vcto.dt_limite_negociacao ASC) as NUM_REUNIAO
from dl_gestao_recursos.ajuste_bmf pco
left join dl_Gestao_recursos.sac_fu_vcto vcto on pco.fuvct_fuatv_cd = vcto.fuvct_fuatv_cd and pco.fuvct_cd = vcto.fuvct_cd
where pco.fuvct_fuatv_cd like '%CPM%' and pco.dt between  '{data_inicial}' and '{data_final}'

--and pco.vl_preco > 3
order by pco.dt asc , vcto.dt_limite_negociacao asc , vcto.vl_preco_exerc asc
 """


df_cpm = pd.read_sql(Select, session) 
df_cpm.DT = pd.to_datetime(df_cpm.DT, format='%Y-%m-%d') 
df_cpm['DT_TXT'] = df_cpm['DT'].dt.strftime('%d/%m/%Y')
pd.set_option('mode.chained_assignment', None)

In [3]:
df_cpm = pd.read_sql(Select, session) 
df_cpm.DT = pd.to_datetime(df_cpm.DT, format='%Y-%m-%d') 
df_cpm['DT_TXT'] = df_cpm['DT'].dt.strftime('%d/%m/%Y')
pd.set_option('mode.chained_assignment', None)

In [4]:
display(df_cpm)

,DT,CONTRATO,FUVCT_CD,VL_PRECO,VL_AJUSTE_DIA_ANTERIOR,DT_LIMITE_NEGOCIACAO,Mes_reuniao,CUT_HIKE,CUT_HIKE_2,ROTULO,teste,NUM_REUNIAO,DT_TXT
0,2024-07-22,CPMNS80,NS80,1.000000e-07,1.000000e-07,2024-08-01,8,-100.0,- 100,Aug-24 - 100,-,1.0,22/07/2024
1,2024-07-22,CPMNS81,NS81,1.000000e-07,1.000000e-07,2024-08-01,8,-75.0,- 75,Aug-24 - 75,-,1.0,22/07/2024
2,2024-07-22,CPMNS82,NS82,1.000000e-07,1.000000e-07,2024-08-01,8,-50.0,- 50,Aug-24 - 50,-,1.0,22/07/2024
3,2024-07-22,CPMNS83,NS83,1.000000e-07,1.000000e-07,2024-08-01,8,-25.0,- 25,Aug-24 - 25,-,1.0,22/07/2024
4,2024-07-22,CPMNS84,NS84,9.300000e+01,9.350000e+01,2024-08-01,8,0.0,0,Aug-24 0,,1.0,22/07/2024
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1633,2024-10-11,CPMFT84,FT84,1.000000e-07,1.000000e-07,2025-01-30,1,0.0,0,Jan-25 0,,3.0,11/10/2024
1634,2024-10-11,CPMFT85,FT85,1.000000e-07,1.000000e-07,2025-01-30,1,25.0,+ 25,Jan-25 + 25,+,3.0,11/10/2024
1635,2024-10-11,CPMFT86,FT86,1.000000e-07,1.000000e-07,2025-01-30,1,50.0,+ 50,Jan-25 + 50,+,3.0,11/10/2024
1636,2024-10-11,CPMFT87,FT87,1.000000e-07,1.000000e-07,2025-01-30,1,75.0,+ 75,Jan-25 + 75,+,3.0,11/10/2024


In [5]:
vis_spec = r"""{"config":[{"config":{"defaultAggregated":true,"geoms":["bar"],"coordSystem":"generic","limit":-1,"timezoneDisplayOffset":0},"encodings":{"dimensions":[{"fid":"DT","name":"DT","basename":"DT","semanticType":"temporal","analyticType":"dimension","offset":0},{"fid":"CONTRATO","name":"CONTRATO","basename":"CONTRATO","semanticType":"nominal","analyticType":"dimension","offset":0},{"fid":"FUVCT_CD","name":"FUVCT_CD","basename":"FUVCT_CD","semanticType":"nominal","analyticType":"dimension","offset":0},{"fid":"DT_LIMITE_NEGOCIACAO","name":"DT_LIMITE_NEGOCIACAO","basename":"DT_LIMITE_NEGOCIACAO","semanticType":"temporal","analyticType":"dimension","offset":0},{"fid":"Mes_reuniao","name":"Mes_reuniao","basename":"Mes_reuniao","semanticType":"nominal","analyticType":"dimension","offset":0},{"fid":"CUT_HIKE_2","name":"CUT_HIKE_2","basename":"CUT_HIKE_2","semanticType":"nominal","analyticType":"dimension","offset":0},{"fid":"ROTULO","name":"ROTULO","basename":"ROTULO","semanticType":"nominal","analyticType":"dimension","offset":0},{"fid":"teste","name":"teste","basename":"teste","semanticType":"nominal","analyticType":"dimension","offset":0},{"fid":"gw_mea_key_fid","name":"Measure names","analyticType":"dimension","semanticType":"nominal"}],"measures":[{"fid":"VL_PRECO","name":"VL_PRECO","basename":"VL_PRECO","analyticType":"measure","semanticType":"quantitative","aggName":"sum","offset":0},{"fid":"VL_AJUSTE_DIA_ANTERIOR","name":"VL_AJUSTE_DIA_ANTERIOR","basename":"VL_AJUSTE_DIA_ANTERIOR","analyticType":"measure","semanticType":"quantitative","aggName":"sum","offset":0},{"fid":"CUT_HIKE","name":"CUT_HIKE","basename":"CUT_HIKE","analyticType":"measure","semanticType":"quantitative","aggName":"sum","offset":0},{"fid":"NUM_REUNIAO","name":"NUM_REUNIAO","basename":"NUM_REUNIAO","analyticType":"measure","semanticType":"quantitative","aggName":"sum","offset":0},{"fid":"gw_count_fid","name":"Row count","analyticType":"measure","semanticType":"quantitative","aggName":"sum","computed":true,"expression":{"op":"one","params":[],"as":"gw_count_fid"}},{"fid":"gw_mea_val_fid","name":"Measure values","analyticType":"measure","semanticType":"quantitative","aggName":"sum"}],"rows":[{"fid":"VL_PRECO","name":"VL_PRECO","basename":"VL_PRECO","analyticType":"measure","semanticType":"quantitative","aggName":"sum","offset":0}],"columns":[{"fid":"DT","name":"DT","basename":"DT","semanticType":"temporal","analyticType":"dimension","offset":0},{"fid":"Mes_reuniao","name":"Mes_reuniao","basename":"Mes_reuniao","semanticType":"nominal","analyticType":"dimension","offset":0}],"color":[{"fid":"CUT_HIKE_2","name":"CUT_HIKE_2","basename":"CUT_HIKE_2","semanticType":"nominal","analyticType":"dimension","offset":0}],"opacity":[],"size":[],"shape":[],"radius":[],"theta":[],"longitude":[],"latitude":[],"geoId":[],"details":[],"filters":[{"fid":"DT","name":"DT","basename":"DT","semanticType":"temporal","analyticType":"dimension","offset":0,"rule":{"type":"one of","value":["2024-08-05","2024-07-31","2024-08-02","2024-07-17"]}},{"fid":"CUT_HIKE_2","name":"CUT_HIKE_2","basename":"CUT_HIKE_2","semanticType":"nominal","analyticType":"dimension","offset":0,"rule":{"type":"one of","value":["   0","+  50","+  25"]}},{"fid":"DT_LIMITE_NEGOCIACAO","name":"DT_LIMITE_NEGOCIACAO","basename":"DT_LIMITE_NEGOCIACAO","semanticType":"temporal","analyticType":"dimension","offset":0,"rule":{"type":"one of","value":["2024-11-07","2024-09-19","2024-12-12"]}}],"text":[]},"layout":{"showActions":false,"showTableSummary":false,"stack":"stack","interactiveScale":false,"zeroScale":true,"size":{"mode":"auto","width":320,"height":200},"format":{},"geoKey":"name","resolve":{"x":false,"y":false,"color":false,"opacity":false,"shape":false,"size":false}},"visId":"gw_W8KS","name":"Chart 1"},{"config":{"defaultAggregated":true,"geoms":["bar"],"coordSystem":"generic","limit":-1,"timezoneDisplayOffset":0},"encodings":{"dimensions":[{"fid":"DT","name":"DT","basename":"DT","semanticType":"temporal","analyticType":"dimension","offset":0},{"fid":"CONTRATO","name":"CONTRATO","basename":"CONTRATO","semanticType":"nominal","analyticType":"dimension","offset":0},{"fid":"FUVCT_CD","name":"FUVCT_CD","basename":"FUVCT_CD","semanticType":"nominal","analyticType":"dimension","offset":0},{"fid":"DT_LIMITE_NEGOCIACAO","name":"DT_LIMITE_NEGOCIACAO","basename":"DT_LIMITE_NEGOCIACAO","semanticType":"temporal","analyticType":"dimension","offset":0},{"fid":"Mes_reuniao","name":"Mes_reuniao","basename":"Mes_reuniao","semanticType":"nominal","analyticType":"dimension","offset":0},{"fid":"CUT_HIKE_2","name":"CUT_HIKE_2","basename":"CUT_HIKE_2","semanticType":"nominal","analyticType":"dimension","offset":0},{"fid":"ROTULO","name":"ROTULO","basename":"ROTULO","semanticType":"nominal","analyticType":"dimension","offset":0},{"fid":"teste","name":"teste","basename":"teste","semanticType":"nominal","analyticType":"dimension","offset":0},{"fid":"gw_mea_key_fid","name":"Measure names","analyticType":"dimension","semanticType":"nominal"}],"measures":[{"fid":"VL_PRECO","name":"VL_PRECO","basename":"VL_PRECO","analyticType":"measure","semanticType":"quantitative","aggName":"sum","offset":0},{"fid":"VL_AJUSTE_DIA_ANTERIOR","name":"VL_AJUSTE_DIA_ANTERIOR","basename":"VL_AJUSTE_DIA_ANTERIOR","analyticType":"measure","semanticType":"quantitative","aggName":"sum","offset":0},{"fid":"CUT_HIKE","name":"CUT_HIKE","basename":"CUT_HIKE","analyticType":"measure","semanticType":"quantitative","aggName":"sum","offset":0},{"fid":"NUM_REUNIAO","name":"NUM_REUNIAO","basename":"NUM_REUNIAO","analyticType":"measure","semanticType":"quantitative","aggName":"sum","offset":0},{"fid":"gw_count_fid","name":"Row count","analyticType":"measure","semanticType":"quantitative","aggName":"sum","computed":true,"expression":{"op":"one","params":[],"as":"gw_count_fid"}},{"fid":"gw_mea_val_fid","name":"Measure values","analyticType":"measure","semanticType":"quantitative","aggName":"sum"}],"rows":[{"fid":"VL_PRECO","name":"VL_PRECO","basename":"VL_PRECO","analyticType":"measure","semanticType":"quantitative","aggName":"sum","offset":0}],"columns":[{"fid":"Mes_reuniao","name":"Mes_reuniao","basename":"Mes_reuniao","semanticType":"nominal","analyticType":"dimension","offset":0},{"fid":"DT","name":"DT","basename":"DT","semanticType":"temporal","analyticType":"dimension","offset":0}],"color":[{"fid":"CUT_HIKE_2","name":"CUT_HIKE_2","basename":"CUT_HIKE_2","semanticType":"nominal","analyticType":"dimension","offset":0}],"opacity":[],"size":[],"shape":[],"radius":[],"theta":[],"longitude":[],"latitude":[],"geoId":[],"details":[],"filters":[{"fid":"DT","name":"DT","basename":"DT","semanticType":"temporal","analyticType":"dimension","offset":0,"rule":{"type":"not in","value":[]}},{"fid":"CUT_HIKE_2","name":"CUT_HIKE_2","basename":"CUT_HIKE_2","semanticType":"nominal","analyticType":"dimension","offset":0,"rule":{"type":"one of","value":["   0","+  50","+  25"]}},{"fid":"DT_LIMITE_NEGOCIACAO","name":"DT_LIMITE_NEGOCIACAO","basename":"DT_LIMITE_NEGOCIACAO","semanticType":"temporal","analyticType":"dimension","offset":0,"rule":{"type":"one of","value":["2024-11-07","2024-09-19","2024-12-12"]}}],"text":[]},"layout":{"showActions":false,"showTableSummary":false,"stack":"stack","interactiveScale":false,"zeroScale":true,"size":{"mode":"auto","width":320,"height":200},"format":{},"geoKey":"name","resolve":{"x":false,"y":false,"color":false,"opacity":false,"shape":false,"size":false}},"visId":"gw_mOdr","name":"Chart 1 Copy"},{"config":{"defaultAggregated":true,"geoms":["auto"],"coordSystem":"generic","limit":-1,"timezoneDisplayOffset":0},"encodings":{"dimensions":[{"fid":"DT","name":"DT","basename":"DT","semanticType":"temporal","analyticType":"dimension","offset":0},{"fid":"CONTRATO","name":"CONTRATO","basename":"CONTRATO","semanticType":"nominal","analyticType":"dimension","offset":0},{"fid":"FUVCT_CD","name":"FUVCT_CD","basename":"FUVCT_CD","semanticType":"nominal","analyticType":"dimension","offset":0},{"fid":"DT_LIMITE_NEGOCIACAO","name":"DT_LIMITE_NEGOCIACAO","basename":"DT_LIMITE_NEGOCIACAO","semanticType":"temporal","analyticType":"dimension","offset":0},{"fid":"Mes_reuniao","name":"Mes_reuniao","basename":"Mes_reuniao","semanticType":"nominal","analyticType":"dimension","offset":0},{"fid":"CUT_HIKE_2","name":"CUT_HIKE_2","basename":"CUT_HIKE_2","semanticType":"nominal","analyticType":"dimension","offset":0},{"fid":"ROTULO","name":"ROTULO","basename":"ROTULO","semanticType":"nominal","analyticType":"dimension","offset":0},{"fid":"teste","name":"teste","basename":"teste","semanticType":"nominal","analyticType":"dimension","offset":0},{"fid":"gw_mea_key_fid","name":"Measure names","analyticType":"dimension","semanticType":"nominal"}],"measures":[{"fid":"VL_PRECO","name":"VL_PRECO","basename":"VL_PRECO","analyticType":"measure","semanticType":"quantitative","aggName":"sum","offset":0},{"fid":"VL_AJUSTE_DIA_ANTERIOR","name":"VL_AJUSTE_DIA_ANTERIOR","basename":"VL_AJUSTE_DIA_ANTERIOR","analyticType":"measure","semanticType":"quantitative","aggName":"sum","offset":0},{"fid":"CUT_HIKE","name":"CUT_HIKE","basename":"CUT_HIKE","analyticType":"measure","semanticType":"quantitative","aggName":"sum","offset":0},{"fid":"NUM_REUNIAO","name":"NUM_REUNIAO","basename":"NUM_REUNIAO","analyticType":"measure","semanticType":"quantitative","aggName":"sum","offset":0},{"fid":"gw_count_fid","name":"Row count","analyticType":"measure","semanticType":"quantitative","aggName":"sum","computed":true,"expression":{"op":"one","params":[],"as":"gw_count_fid"}},{"fid":"gw_mea_val_fid","name":"Measure values","analyticType":"measure","semanticType":"quantitative","aggName":"sum"}],"rows":[],"columns":[],"color":[],"opacity":[],"size":[],"shape":[],"radius":[],"theta":[],"longitude":[],"latitude":[],"geoId":[],"details":[],"filters":[],"text":[]},"layout":{"showActions":false,"showTableSummary":false,"stack":"stack","interactiveScale":false,"zeroScale":true,"size":{"mode":"auto","width":320,"height":200},"format":{},"geoKey":"name","resolve":{"x":true,"y":true,"color":true,"opacity":true,"shape":true,"size":true},"scaleIncludeUnmatchedChoropleth":false,"showAllGeoshapeInChoropleth":false,"colorPalette":"","useSvg":false,"scale":{"opacity":{},"size":{}}},"visId":"gw_JLEB","name":"Chart 3"},{"config":{"defaultAggregated":true,"geoms":["line"],"coordSystem":"generic","limit":-1,"timezoneDisplayOffset":0},"encodings":{"dimensions":[{"fid":"DT","name":"DT","basename":"DT","semanticType":"temporal","analyticType":"dimension","offset":0},{"fid":"CONTRATO","name":"CONTRATO","basename":"CONTRATO","semanticType":"nominal","analyticType":"dimension","offset":0},{"fid":"FUVCT_CD","name":"FUVCT_CD","basename":"FUVCT_CD","semanticType":"nominal","analyticType":"dimension","offset":0},{"fid":"DT_LIMITE_NEGOCIACAO","name":"DT_LIMITE_NEGOCIACAO","basename":"DT_LIMITE_NEGOCIACAO","semanticType":"temporal","analyticType":"dimension","offset":0},{"fid":"Mes_reuniao","name":"Mes_reuniao","basename":"Mes_reuniao","semanticType":"nominal","analyticType":"dimension","offset":0},{"fid":"CUT_HIKE_2","name":"CUT_HIKE_2","basename":"CUT_HIKE_2","semanticType":"nominal","analyticType":"dimension","offset":0},{"fid":"ROTULO","name":"ROTULO","basename":"ROTULO","semanticType":"nominal","analyticType":"dimension","offset":0},{"fid":"teste","name":"teste","basename":"teste","semanticType":"nominal","analyticType":"dimension","offset":0},{"fid":"gw_mea_key_fid","name":"Measure names","analyticType":"dimension","semanticType":"nominal"}],"measures":[{"fid":"VL_PRECO","name":"VL_PRECO","basename":"VL_PRECO","analyticType":"measure","semanticType":"quantitative","aggName":"sum","offset":0},{"fid":"VL_AJUSTE_DIA_ANTERIOR","name":"VL_AJUSTE_DIA_ANTERIOR","basename":"VL_AJUSTE_DIA_ANTERIOR","analyticType":"measure","semanticType":"quantitative","aggName":"sum","offset":0},{"fid":"CUT_HIKE","name":"CUT_HIKE","basename":"CUT_HIKE","analyticType":"measure","semanticType":"quantitative","aggName":"sum","offset":0},{"fid":"NUM_REUNIAO","name":"NUM_REUNIAO","basename":"NUM_REUNIAO","analyticType":"measure","semanticType":"quantitative","aggName":"sum","offset":0},{"fid":"gw_count_fid","name":"Row count","analyticType":"measure","semanticType":"quantitative","aggName":"sum","computed":true,"expression":{"op":"one","params":[],"as":"gw_count_fid"}},{"fid":"gw_mea_val_fid","name":"Measure values","analyticType":"measure","semanticType":"quantitative","aggName":"sum"}],"rows":[{"fid":"VL_PRECO","name":"VL_PRECO","basename":"VL_PRECO","analyticType":"measure","semanticType":"quantitative","aggName":"sum","offset":0}],"columns":[{"fid":"CUT_HIKE_2","name":"CUT_HIKE_2","basename":"CUT_HIKE_2","semanticType":"nominal","analyticType":"dimension","offset":0},{"fid":"DT","name":"DT","basename":"DT","semanticType":"temporal","analyticType":"dimension","offset":0}],"color":[{"fid":"Mes_reuniao","name":"Mes_reuniao","basename":"Mes_reuniao","semanticType":"nominal","analyticType":"dimension","offset":0}],"opacity":[],"size":[],"shape":[],"radius":[],"theta":[],"longitude":[],"latitude":[],"geoId":[],"details":[],"filters":[{"fid":"Mes_reuniao","name":"Mes_reuniao","basename":"Mes_reuniao","semanticType":"nominal","analyticType":"dimension","offset":0,"rule":{"type":"one of","value":["         11","          9","         12"]}},{"fid":"CUT_HIKE_2","name":"CUT_HIKE_2","basename":"CUT_HIKE_2","semanticType":"nominal","analyticType":"dimension","offset":0,"rule":{"type":"one of","value":["   0","+  50","+  25"]}}],"text":[]},"layout":{"showActions":false,"showTableSummary":false,"stack":"stack","interactiveScale":false,"zeroScale":true,"size":{"mode":"auto","width":792,"height":297},"format":{},"geoKey":"name","resolve":{"x":false,"y":false,"color":false,"opacity":false,"shape":false,"size":false}},"visId":"gw_2H0I","name":"Chart 1 Copy Copy"}],"chart_map":{},"workflow_list":[{"workflow":[{"type":"filter","filters":[{"fid":"DT","rule":{"type":"one of","value":["2024-08-05","2024-07-31","2024-08-02","2024-07-17"]}},{"fid":"CUT_HIKE_2","rule":{"type":"one of","value":["   0","+  50","+  25"]}},{"fid":"DT_LIMITE_NEGOCIACAO","rule":{"type":"one of","value":["2024-11-07","2024-09-19","2024-12-12"]}}]},{"type":"view","query":[{"op":"aggregate","groupBy":["DT","Mes_reuniao","CUT_HIKE_2"],"measures":[{"field":"VL_PRECO","agg":"sum","asFieldKey":"VL_PRECO_sum"}]}]}]},{"workflow":[{"type":"filter","filters":[{"fid":"DT","rule":{"type":"not in","value":[]}},{"fid":"CUT_HIKE_2","rule":{"type":"one of","value":["   0","+  50","+  25"]}},{"fid":"DT_LIMITE_NEGOCIACAO","rule":{"type":"one of","value":["2024-11-07","2024-09-19","2024-12-12"]}}]},{"type":"view","query":[{"op":"aggregate","groupBy":["Mes_reuniao","DT","CUT_HIKE_2"],"measures":[{"field":"VL_PRECO","agg":"sum","asFieldKey":"VL_PRECO_sum"}]}]}]},{"workflow":[{"type":"view","query":[{"op":"raw","fields":[]}]}]},{"workflow":[{"type":"filter","filters":[{"fid":"Mes_reuniao","rule":{"type":"one of","value":["         11","          9","         12"]}},{"fid":"CUT_HIKE_2","rule":{"type":"one of","value":["   0","+  50","+  25"]}}]},{"type":"view","query":[{"op":"aggregate","groupBy":["CUT_HIKE_2","DT","Mes_reuniao"],"measures":[{"field":"VL_PRECO","agg":"sum","asFieldKey":"VL_PRECO_sum"}]}]}]}],"version":"0.4.9.1"}"""
pyg.walk(df_cpm, spec=vis_spec)


Box(children=(HTML(value='\n<div id="ifr-pyg-0006247199849516AOkNYvc6iyVfD5mj" style="height: auto">\n    <hea…

ConversionException: Conversion Error: Could not convert string '2024-08-05' to INT64

In [ ]:
from pivottablejs import pivot_ui
pivot_ui(df_cpm) 